# Lección 12 - Reducción del Historial de Chat con Bloc de Notas del Agente

Este cuaderno demuestra cómo gestionar el contexto en conversaciones largas usando Microsoft Agent Framework. A medida que las conversaciones crecen, el conteo de tokens aumenta — eventualmente excediendo la ventana de contexto del modelo. Abordamos esto con un **patrón de resumen de contexto** y un **bloc de notas del agente** para memoria persistente.

## Lo que aprenderás:
1. **Por qué importa la gestión del contexto**: Comprender los límites de tokens y las ventanas de contexto
2. **Agentes conscientes del contexto**: Construir agentes que gestionen su propio contexto de conversación
3. **Patrón de resumen de contexto**: Usar herramientas para condensar el historial de conversación
4. **Bloc de notas del agente**: Memoria persistente que sobrevive la reducción del contexto

## Requisitos previos:
- Configuración de Azure OpenAI con variables de entorno configuradas
- Comprensión de conceptos básicos de agentes de lecciones anteriores


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Por qué importa la gestión del contexto

Cada LLM tiene una **ventana de contexto** finita: el número máximo de tokens que puede procesar en una sola solicitud. A medida que progresa una conversación de múltiples turnos:

- El **conteo de tokens crece linealmente** con cada mensaje del usuario y respuesta del asistente.
- Los **tokens del prompt dominan el costo** porque todo el historial se reenvía en cada turno.
- Eventualmente la conversación **excede la ventana de contexto** y el modelo o bien trunca o da error.

### Estrategias para gestionar el contexto

| Estrategia | Cómo funciona | Compromiso |
|---|---|---|
| **Truncamiento** | Eliminar los mensajes más antiguos | Pierde contexto inicial |
| **Resumen** | Condensar mensajes antiguos en un resumen | Se pierde algo de detalle, pero se conservan puntos clave |
| **Bloc de notas / Memoria externa** | Almacenar hechos clave fuera de la conversación | Requiere llamadas a herramientas, pero sobrevive a cualquier reducción |

En este cuaderno combinamos el **resumen** con una **herramienta de bloc de notas** para que el agente pueda mantener la continuidad incluso cuando el historial de la conversación esté condensado.


## Creando un Agente Consciente del Contexto


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulando una conversación larga

Recorramos una conversación de múltiples turnos para ver cómo se acumula el contexto. El agente debe retener detalles clave (preferencias, presupuesto, fechas de viaje) a lo largo de los turnos y demostrar continuidad.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Observa cómo el agente retiene el contexto de turnos anteriores: sabe sobre Japón, sushi, templos, fotografía, el presupuesto de $3000, viajes en solitario y el período de abril. En una conversación corta esto funciona bien, pero a medida que la conversación crece, enviar todo el historial resulta costoso.

Continuemos la conversación con más turnos para ver cómo se acumula el contexto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

As the conversation grows, we can use a **summarization tool** to condense accumulated context into a compact format. The agent calls this tool to record key preferences so that even if older messages are dropped, the essential information is preserved.

This pattern is the building block for more sophisticated history reduction:
1. The agent identifies key facts from the conversation
2. It calls the summarization tool to persist them
3. Older messages can be safely removed because the summary captures what matters

Below we define a `summarize_preferences` tool that the agent can call to record a compact summary of what it has learned.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Resumen

En esta lección aprendiste cómo gestionar el contexto en conversaciones de agentes de larga duración utilizando Microsoft Agent Framework:

### Conceptos Clave
- **Las ventanas de contexto son finitas** — cada token en el historial de la conversación cuesta dinero y cuenta para el límite.
- **Las herramientas de resumen** permiten al agente condensar el contexto acumulado en resúmenes compactos, reduciendo el uso de tokens mientras se preserva la información esencial.
- **Las libretas del agente (scratchpads)** proporcionan memoria externa persistente que sobrevive a cualquier reducción de la conversación.

### Lo que Construiste
- Un **agente consciente del contexto** que mantiene la continuidad a través de conversaciones de múltiples turnos
- Una **herramienta de resumen** (`summarize_preferences`) que registra detalles clave del usuario en un formato compacto
- Una **conversación de múltiples turnos** que demuestra la retención del contexto y la gestión de cambios

### Aplicaciones en el Mundo Real
- **Bots de Servicio al Cliente**: recordar preferencias durante largas sesiones de soporte
- **Asistentes Personales**: hacer seguimiento de proyectos en curso sin tener que reexplicar el contexto
- **Tutores Educativos**: mantener el progreso del estudiante a lo largo de muchas interacciones

### Próximos Pasos
- Implementar una herramienta completa de libreta con persistencia basada en archivos
- Añadir truncamiento automático del historial después del resumen
- Combinar con bases de datos vectoriales para búsqueda semántica en memoria
- Construir agentes que puedan reanudar conversaciones días después con contexto completo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
